GSESolver_logo_black_short.png

**A FAIR ML workflow to determine the plausibility of using the General Solubility Equation for small molecules**

Intrinsic solubility (expressed as $\log{S_0}$) indicates the maximum concentration of a compound in its neutral state that can be dissolved in water. This physicochemical property is evergrowingly assessed in drug design/developement, medicinal chemistry, and agrochemical sciences.

It is often not possible to experimentally measure the $\log{S_0}$. Thus, the General Solubility Equation (**GSE**) is oftentimes used for a rapid assessment, which uses the molecule's melting point ($mp$ in °C) and *n*-octanol/water partition coefficient ($\log{P_{\text{N}}}$):

$$\log{S_0}=0.5-0.01(mp-25)-\log{P_{\text{N}}}$$

*When should I use the GSE?* This notebook provides an Artificial Neural Network model that assesses if your molecules of interest might have an accurate $\log{S_0}$ prediction using the GSE.

---

 **WARNING:** *This model was created for small organic drug-like molecules. Unaccurate results might occur if large molecules (M > 1000 Da), salts, organometallic complexes, or highly halogenated molecules are evaluated.*

**RECENT UPDATES:**

23/02/2026 *(Added GSESolver logo and description)*

---


##**1. Tools Instalation**

These cells do not need any imput. You just have to run them.

You can run the cells by pressing the *play* button, or by pressing `ctrl + Enter`.


---

This first cell will install an appropriate version of `numpy`. It might restart your kernel. You just need to **press the "restart" button** and keep going.


In [ ]:
#install appropriate version of numpy
!pip uninstall numpy -y
!pip install numpy==1.26.4

This cell will check the current `numpy` version of this notebook. If it says `1.26.4`, everything is alright. If otherwise, re-run the previous cell.

In [ ]:
import numpy as np
print(np.__version__)

This next cell will restart your session. After restart, run this cell again, and the output should say "`✨🍰✨ Everything looks OK!`". If so, you can keep going.

In [ ]:
!pip install -q condacolab
import condacolab
condacolab.install_from_url("https://github.com/conda-forge/miniforge/releases/download/25.3.1-0/Miniforge3-Linux-x86_64.sh")

Then, run these next cells. It might take a couple of minutes (ca. 4 minutes).

In [ ]:
%%capture
!pip install rdkit
!conda install openbabel -y
!pip install jazzy
!conda install conda-forge::r-rcdk==3.8.1 -y
!conda install bioconda::bioconductor-chemminer==3.54.0 -y
!conda install -c conda-forge r-dplyr==1.1.4

This cell will bring all the relevant script and databases for GSE ML models from our GitHub repo. No input is required.

In [ ]:
!git clone https://github.com/cbio3lab/GSESolver.git
!cp /content/GSESolver/DATABASES/RFE_SolCbio3_descs_fps.csv /content/
!cp /content/GSESolver/SCRIPTS/ML_test/nn_model.keras /content/
!cp -r GSESolver/SCRIPTS/descriptors/* /content/
!rm -r CalculateDescriptors
!rm -rf GSESolver

Keep running...

In [ ]:
import pandas as pd
from calculate_stereocenters import calculate_stereocenters

df_descs = pd.read_csv('RFE_SolCbio3_descs_fps.csv')
df_descs.drop(columns=['smiles','dev'], inplace=True)

#calculate descriptors function
from calculateDescriptors import calculateDescriptors

def create_descriptors_table(smiles):
    if len(smiles) == 0:
      return
    unique_smiles = len(smiles) == 1
    if unique_smiles:
      smiles = smiles + ["C"]
    descriptors = calculateDescriptors(smiles)
    if unique_smiles:
      descriptors.drop(descriptors.tail(1).index,inplace = True)
      smiles = smiles[:-1]
    return descriptors



#calculate fingerprints function
from morgan_fingerprint_generator import generate_morgan_fingerprints

def create_fp_table(smiles):
    if len(smiles) == 0:
      return
    unique_smiles = len(smiles) == 1
    if unique_smiles:
      smiles = smiles + ["C"]
    fps = generate_morgan_fingerprints(smiles)
    if unique_smiles:
      fps.drop(fps.tail(1).index,inplace = True)
      smiles = smiles[:-1]
    return fps



---


## **2. Import your molecules**


### **2.1. Insert your Molecules**


In the following cells, you will input your molecules by running the following cell. A text box will appear:


*   `Enter the SMILES of your molecules:`
Herein, you will paste the SMILES code of your molecule(s). If you want to input multiple molecules, you have to enter each SMILES code separated by spaces.

In [ ]:
%%capture

from google.colab import data_table
data_table.enable_dataframe_formatter()
smiles = input("Enter the SMILES of your molecules:\n")
smiles = smiles.split(" ")
descriptors = create_descriptors_table(smiles) #calculate descriptors from SMILES
fingerprint_df = generate_morgan_fingerprints(smiles)
df = pd.concat([descriptors, fingerprint_df], axis=1)
df['NumAtomStereoCenters_rdkit'] = calculate_stereocenters(smiles)

# Get the list of column names from df_descs
columns_in_df_descs = df_descs.columns

# Filter df to keep only the columns that are present in both df and df_descs
df = df[df.columns.intersection(columns_in_df_descs)]

##**2.2. Or else, insert your data in a `.xlsx` file**

Otherwise, you can input your data as a `.xlsx` file.

Your file must contain this column:

* **`SMILES`:** The SMILES code for each molecules.

First, upload your `.xlsx` file into this colab in the "files" menu on the left of the screen.

Then, run the next cell and type the name of your `.xlsx` file.

In [ ]:
%%capture
from google.colab import data_table

data_table.enable_dataframe_formatter()
fileName = input("Enter the name of the file: ") #enter the name of your file


#read the SMILES codes
fileFound = False
try:
  excel = pd.read_excel(fileName)
  fileFound = True
except FileNotFoundError:
  print("File not found")

smiles_col_name = None
for col in excel.columns:
  if col.upper() == "SMILES":
    smiles_col_name = col
    break

if smiles_col_name is None:
    raise KeyError("Column 'SMILES' not found in the Excel file.")

#calculate descriptors
descriptors = create_descriptors_table(excel[smiles_col_name].tolist())
fingerprint_df = generate_morgan_fingerprints(excel[smiles_col_name].tolist())
df = pd.concat([descriptors, fingerprint_df], axis=1)
df['NumAtomStereoCenters_rdkit'] = calculate_stereocenters(excel[smiles_col_name])


# Get the list of column names from df_descs
columns_in_df_descs = df_descs.columns

# Filter df to keep only the columns that are present in both df and df_descs
df = df[df.columns.intersection(columns_in_df_descs)]

## **3. Run the models with your molecules**

In the following steps, you will only need to run the cells until the outputs are downloaded to your computer.

The following cell will **evaluate** your molecules into our ML models.

In [ ]:
import tensorflow as tf
from tensorflow import keras
import numpy as np


# Load the NN model
try:
    nn_model = keras.models.load_model('nn_model.keras')
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")


features_for_prediction = df.values # Convert DataFrame to a NumPy array

# Make predictions
try:
    predictions = (nn_model.predict(features_for_prediction)> 0.5).astype(int)
    predictions = np.where(predictions == 0, 'Use GSE', 'Do not use GSE')

    print("Predictions generated successfully.")

    # Add predictions back to your original DataFrame
    # If the model has a single output, `predictions` will likely be a 1D or 2D array (samples, 1).
    # If it has multiple outputs, `predictions` will be a list of arrays.
    if isinstance(predictions, list):
        # Handle multiple outputs if necessary, e.g., assign to multiple columns
        for i, pred_output in enumerate(predictions):
            df[f'prediction_output_{i+1}'] = pred_output
    else:
        # Assuming single output
        df['prediction'] = predictions.flatten() # Flatten if predictions is (n_samples, 1)

except Exception as e:
    print(f"Error during prediction: {e}")
    print("Please ensure the input data 'features_for_prediction' is correctly preprocessed and shaped for the model.")

The following cell will give you the **predictions**. The "prediction" will recommend you which equation is the most appropriate depending on the molecule and desired pH.

A csv file named `GSESolver_results.csv` will be created and automatically downloaded.

In [ ]:
from google.colab import files

output = pd.DataFrame(data=df, columns=['prediction'])

try:
  output.insert(0, 'smiles', smiles)
except NameError:
  print('No manual SMILES inserted, trying with the Excel file.')
  output.insert(0, 'smiles', excel[smiles_col_name])

output.to_csv('GSESolver_results.csv', index=False)
files.download('GSESolver_results.csv')
output

# **If you used this notebook, please read and cite our work**
"Data-driven critical evaluation of the General Solubility Equation: When is it a valid solubility predictor?", *preprint*, **2026**, DOI: